# CI Rehab Voice Cloning Discrimination Experiment: Data Analysis Pipeline

This notebook implements the full analysis pipeline across Stages 1-5:
1. Data ingestion and validation
2. Feature engineering
3. Accuracy and signal detection theory metrics
4. Consistency analysis
5. Factor analysis (statistical modeling)

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from scipy.stats import norm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

DATA_DIR = Path("Data")
assert DATA_DIR.exists(), "Data directory does not exist. Please check the symlink."

## Stage 1: Data Ingestion and Validation

In [ ]:
result_files = sorted(DATA_DIR.glob("AI_VoiceTask_P*_results.csv"))
survey_files = sorted(DATA_DIR.glob("AI_VoiceTask_P*_survey.json"))
checkpoint_files = sorted(DATA_DIR.glob("AI_VoiceTask_P*_checkpoint.json"))

print(f"results files: {len(result_files)}")
print(f"survey files: {len(survey_files)}")
print(f"checkpoint files: {len(checkpoint_files)}")

In [ ]:
def parse_pid_from_name(path: Path) -> str:
    m = re.search(r"AI_VoiceTask_(P\d+)_", path.name)
    if not m:
        raise ValueError(f"Cannot parse pid from filename: {path.name}")
    return m.group(1)

all_results = []
for rf in result_files:
    df = pd.read_csv(rf)
    df["ParticipantID_from_file"] = parse_pid_from_name(rf)
    all_results.append(df)

results_df = pd.concat(all_results, ignore_index=True)
print("combined rows:", len(results_df))
print("participants in results:", results_df["ParticipantID"].nunique())
results_df.head(3)

In [ ]:
survey_records = []
for sf in survey_files:
    with open(sf, "r", encoding="utf-8") as f:
        obj = json.load(f)
    survey_records.append(
        {
            "ParticipantID": obj.get("pid", parse_pid_from_name(sf)),
            "criteria": obj.get("criteria", ""),
            "survey_file": sf.name,
        }
    )

survey_df = pd.DataFrame(survey_records).sort_values("ParticipantID").reset_index(drop=True)
survey_df.head()

In [ ]:
# Stage 1 validation checks
expected_cols = {
    "ParticipantID", "File", "Filename", "Artic", "Speaker", "Model", "Repetition", "IsAI",
    "Confidence", "ReplayCount", "TrialOrder", "IsConsistencyCheck", "TrialStart", "TrialEnd"
}

actual_cols = set(results_df.columns) - {"ParticipantID_from_file"}
missing_cols = expected_cols - actual_cols
extra_cols = actual_cols - expected_cols

print("missing cols:", missing_cols)
print("extra cols:", extra_cols)
print("participants (results):", sorted(results_df["ParticipantID"].unique())[:3], "...", sorted(results_df["ParticipantID"].unique())[-3:])
print("participants count (results):", results_df["ParticipantID"].nunique())
print("participants count (survey):", survey_df["ParticipantID"].nunique())

print("\nValue range checks")
print("Speaker values:", sorted(results_df["Speaker"].dropna().unique()))
print("Model values:", sorted(results_df["Model"].dropna().unique()))
print("IsAI values:", sorted(results_df["IsAI"].dropna().unique()))
print("Confidence values:", sorted(results_df["Confidence"].dropna().unique()))

rows_per_pid = results_df.groupby("ParticipantID").size().sort_values()
print("\nRows per participant (min/max):", rows_per_pid.min(), rows_per_pid.max())
rows_per_pid.head()

## Stage 2: Data Cleaning and Feature Engineering

In [ ]:
analysis_df = results_df.copy()

# Ground truth: Model=1 -> human, Model=2/3 -> AI
analysis_df["IsAI_true"] = analysis_df["Model"].isin([2, 3]).astype(int)
analysis_df["Correct"] = (analysis_df["IsAI"].astype(int) == analysis_df["IsAI_true"]).astype(int)

analysis_df["Confidence_num"] = analysis_df["Confidence"].astype(str).str.extract(r"(\d+)").astype(float)

analysis_df["TrialStart"] = pd.to_datetime(analysis_df["TrialStart"], errors="coerce")
analysis_df["TrialEnd"] = pd.to_datetime(analysis_df["TrialEnd"], errors="coerce")
analysis_df["RT_sec"] = (analysis_df["TrialEnd"] - analysis_df["TrialStart"]).dt.total_seconds()

analysis_df["Topic"] = ((analysis_df["Artic"].astype(int) - 1) // 10 + 1).astype(int)

analysis_df["IsConsistencyCheck"] = analysis_df["IsConsistencyCheck"].astype(str).str.lower().eq("true")

formal_df = analysis_df[~analysis_df["IsConsistencyCheck"]].copy()
consistency_df = analysis_df[analysis_df["IsConsistencyCheck"]].copy()

print("analysis rows:", len(analysis_df))
print("formal rows:", len(formal_df))
print("consistency rows:", len(consistency_df))
analysis_df[["ParticipantID", "Model", "IsAI", "IsAI_true", "Correct", "Confidence", "Confidence_num", "RT_sec", "Topic"]].head()

In [ ]:
print("Topic range:", analysis_df["Topic"].min(), "to", analysis_df["Topic"].max())
print("Missing Confidence_num:", analysis_df["Confidence_num"].isna().sum())
print("Missing RT_sec:", analysis_df["RT_sec"].isna().sum())
print("RT summary:")
analysis_df["RT_sec"].describe()

## Stage 3: Accuracy Analysis (Descriptive Stats + SDT)

In [ ]:
overall_accuracy = formal_df["Correct"].mean()
per_participant_accuracy = formal_df.groupby("ParticipantID", as_index=False)["Correct"].mean().rename(columns={"Correct": "Accuracy"})

print(f"Overall accuracy (formal trials): {overall_accuracy:.3f}")
per_participant_accuracy.head()

In [ ]:
acc_by_speaker = formal_df.groupby("Speaker", as_index=False)["Correct"].mean().rename(columns={"Correct": "Accuracy"})
acc_by_model = formal_df.groupby("Model", as_index=False)["Correct"].mean().rename(columns={"Correct": "Accuracy"})
acc_by_truth = formal_df.groupby("IsAI_true", as_index=False)["Correct"].mean().rename(columns={"Correct": "Accuracy"})
acc_by_truth["TruthLabel"] = acc_by_truth["IsAI_true"].map({0: "Human", 1: "AI"})

print("Accuracy by Speaker")
display(acc_by_speaker)
print("Accuracy by Model")
display(acc_by_model)
print("Accuracy by True Label")
display(acc_by_truth)

In [ ]:
def clipped_rate(rate: float, n: int) -> float:
    # Avoid inf values in z-transform for d-prime when rate is 0 or 1.
    if n <= 0:
        return np.nan
    eps = 0.5 / n
    return min(max(rate, eps), 1 - eps)


def compute_sdt_for_pid(df_pid: pd.DataFrame) -> pd.Series:
    ai_trials = df_pid[df_pid["IsAI_true"] == 1]
    human_trials = df_pid[df_pid["IsAI_true"] == 0]

    hit_rate_raw = (ai_trials["IsAI"] == 1).mean() if len(ai_trials) else np.nan
    fa_rate_raw = (human_trials["IsAI"] == 1).mean() if len(human_trials) else np.nan

    hit_rate = clipped_rate(hit_rate_raw, len(ai_trials))
    fa_rate = clipped_rate(fa_rate_raw, len(human_trials))

    z_hit = norm.ppf(hit_rate) if pd.notna(hit_rate) else np.nan
    z_fa = norm.ppf(fa_rate) if pd.notna(fa_rate) else np.nan

    d_prime = z_hit - z_fa if pd.notna(z_hit) and pd.notna(z_fa) else np.nan
    criterion_c = -0.5 * (z_hit + z_fa) if pd.notna(z_hit) and pd.notna(z_fa) else np.nan

    return pd.Series(
        {
            "n_ai": len(ai_trials),
            "n_human": len(human_trials),
            "hit_rate": hit_rate_raw,
            "false_alarm_rate": fa_rate_raw,
            "d_prime": d_prime,
            "criterion_c": criterion_c,
        }
    )

sdt_per_participant = formal_df.groupby("ParticipantID").apply(compute_sdt_for_pid).reset_index()
display(sdt_per_participant.head())

group_sdt = compute_sdt_for_pid(formal_df)
print("Group-level SDT:")
group_sdt

## Stage 4: Consistency Analysis

In [ ]:
base_formal = formal_df[["ParticipantID", "Filename", "IsAI"]].rename(columns={"IsAI": "IsAI_formal"})
base_check = consistency_df[["ParticipantID", "Filename", "IsAI"]].rename(columns={"IsAI": "IsAI_check"})

consistency_match = base_check.merge(base_formal, on=["ParticipantID", "Filename"], how="left")
consistency_match["Consistent"] = (consistency_match["IsAI_check"] == consistency_match["IsAI_formal"]).astype(float)

consistency_per_pid = (
    consistency_match.groupby("ParticipantID", as_index=False)["Consistent"]
    .mean()
    .rename(columns={"Consistent": "ConsistencyRate"})
)

print("Matched consistency rows:", len(consistency_match))
print("Unmatched consistency rows:", consistency_match["IsAI_formal"].isna().sum())

display(consistency_per_pid.head())
print("Overall consistency rate:", consistency_match["Consistent"].mean())

## Stage 5: Factor Analysis (Exploration + Modeling)

In [ ]:
acc_by_conf = (
    formal_df.groupby("Confidence_num", as_index=False)["Correct"]
    .mean()
    .rename(columns={"Correct": "Accuracy"})
    .sort_values("Confidence_num")
)

acc_by_replay = (
    formal_df.groupby("ReplayCount", as_index=False)["Correct"]
    .mean()
    .rename(columns={"Correct": "Accuracy"})
    .sort_values("ReplayCount")
)

display(acc_by_conf)
display(acc_by_replay)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=acc_by_conf, x="Confidence_num", y="Accuracy", ax=axes[0], color="#4C72B0")
axes[0].set_title("Accuracy by Confidence")
axes[0].set_ylim(0, 1)

sns.lineplot(data=acc_by_replay, x="ReplayCount", y="Accuracy", marker="o", ax=axes[1], color="#55A868")
axes[1].set_title("Accuracy by ReplayCount")
axes[1].set_ylim(0, 1)
plt.tight_layout()

In [ ]:
model_df = formal_df.copy()
model_df["Speaker"] = model_df["Speaker"].astype("category")
model_df["Model"] = model_df["Model"].astype("category")
model_df["ParticipantID"] = model_df["ParticipantID"].astype("category")

# Ensure confidence is numeric for modeling.
if "Confidence_num" not in model_df.columns:
    model_df["Confidence_num"] = (
        model_df["Confidence"].astype(str).str.extract(r"(\d+)").astype(float)
    )
model_df["Confidence_num"] = model_df["Confidence_num"].astype(float)
model_df["ReplayCount"] = model_df["ReplayCount"].astype(float)

# Standardize continuous predictors.
for col in ["Confidence_num", "ReplayCount"]:
    mean_val = model_df[col].mean()
    std_val = model_df[col].std(ddof=0)
    if pd.notna(std_val) and std_val > 0:
        model_df[f"z_{col}"] = (model_df[col] - mean_val) / std_val
    else:
        model_df[f"z_{col}"] = 0.0

formula = "Correct ~ C(Speaker) + C(Model) + z_Confidence_num + z_ReplayCount"

import warnings
import statsmodels.api as sm


def fit_bayes_mixed_robust(dataframe: pd.DataFrame, model_formula: str):
    model = sm.BinomialBayesMixedGLM.from_formula(
        model_formula,
        {"participant_re": "0 + C(ParticipantID)"},
        dataframe,
    )

    # Fit VB mixed model with retry settings.
    vb_attempts = [
        {"fit_method": "BFGS", "minim_opts": {"maxiter": 5000, "gtol": 1e-7}},
        {"fit_method": "L-BFGS-B", "minim_opts": {"maxiter": 8000, "ftol": 1e-9}},
    ]

    last_warning_msgs = []
    last_exception = None
    for kwargs in vb_attempts:
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                vb_result = model.fit_vb(**kwargs)
                warning_msgs = [str(x.message) for x in w]

            if any("did not converge" in msg.lower() for msg in warning_msgs):
                last_warning_msgs = warning_msgs
                continue
            return "VB", vb_result, warning_msgs
        except Exception as vb_err:
            last_exception = vb_err

    # Raise to trigger fallback in caller.
    raise RuntimeError(
        f"VB warnings: {last_warning_msgs}; VB exception: {last_exception}"
    )


# Fit Bayesian mixed logistic model; fallback to GEE if VB fails.
try:
    fit_mode, mixed_result, fit_notes = fit_bayes_mixed_robust(model_df, formula)
    print(f"BinomialBayesMixedGLM fitted successfully with {fit_mode}.")
    if fit_notes:
        print("Fit notes:")
        for note in fit_notes:
            print(" -", note)
    print(mixed_result.summary())
except Exception as e:
    print("BinomialBayesMixedGLM did not converge robustly; fallback to GEE.")
    print("Reason:", e)
    gee_model = smf.gee(
        formula,
        groups="ParticipantID",
        data=model_df,
        family=sm.families.Binomial(),
    )
    gee_result = gee_model.fit()
    print(gee_result.summary())

In [ ]:
# Interaction model: Speaker x Model.
formula_interaction = "Correct ~ C(Speaker) * C(Model) + z_Confidence_num + z_ReplayCount"

# Use the same fitting strategy as the main model.
try:
    inter_fit_mode, inter_result, inter_fit_notes = fit_bayes_mixed_robust(model_df, formula_interaction)
    print(f"Interaction model fitted successfully with {inter_fit_mode}.")
    if inter_fit_notes:
        print("Fit notes:")
        for note in inter_fit_notes:
            print(" -", note)
    print(inter_result.summary())
except Exception as e:
    print("Interaction Bayesian mixed model did not converge robustly; fallback to GEE.")
    print("Reason:", e)
    inter_gee_model = smf.gee(
        formula_interaction,
        groups="ParticipantID",
        data=model_df,
        family=sm.families.Binomial(),
    )
    inter_gee_result = inter_gee_model.fit()
    print(inter_gee_result.summary())

## Export Key Results (Optional)

In [ ]:
from datetime import datetime
import shutil

base_output_dir = Path("outputs")
runs_dir = base_output_dir / "runs"
runs_dir.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = runs_dir / run_timestamp
output_dir.mkdir(parents=True, exist_ok=False)

per_participant_accuracy.to_csv(output_dir / "per_participant_accuracy.csv", index=False)
sdt_per_participant.to_csv(output_dir / "sdt_per_participant.csv", index=False)
consistency_per_pid.to_csv(output_dir / "consistency_per_participant.csv", index=False)
acc_by_speaker.to_csv(output_dir / "accuracy_by_speaker.csv", index=False)
acc_by_model.to_csv(output_dir / "accuracy_by_model.csv", index=False)
acc_by_conf.to_csv(output_dir / "accuracy_by_confidence.csv", index=False)
acc_by_replay.to_csv(output_dir / "accuracy_by_replaycount.csv", index=False)

survey_df.to_csv(output_dir / "survey_criteria.csv", index=False)

latest_dir = base_output_dir / "latest"
if latest_dir.exists() or latest_dir.is_symlink():
    if latest_dir.is_symlink() or latest_dir.is_file():
        latest_dir.unlink()
    else:
        shutil.rmtree(latest_dir)

try:
    latest_dir.symlink_to(output_dir.resolve(), target_is_directory=True)
    latest_note = f"latest symlink: {latest_dir.resolve()} -> {output_dir.resolve()}"
except OSError:
    # Fallback when symlink creation is unavailable.
    shutil.copytree(output_dir, latest_dir)
    latest_note = f"latest copied directory: {latest_dir.resolve()}"

print(f"Exported tables to run directory: {output_dir.resolve()}")
print(latest_note)

## Extended Analysis: Institution, Familiarity, Demographics, Topic, and AI Familiarity

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = Path("Data")

# Load trial-level files
result_files = sorted(DATA_DIR.glob("AI_VoiceTask_P*_results.csv"))
trial_df = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)

# Core trial features
trial_df["ParticipantID"] = trial_df["ParticipantID"].astype(str)
trial_df["ParticipantNum"] = trial_df["ParticipantID"].str.extract(r"(\d+)").astype(int)
trial_df["Institution"] = np.where(trial_df["ParticipantNum"].between(1, 25), "Institution 1", "Institution 2")
trial_df["Model"] = trial_df["Model"].astype(int)
trial_df["Speaker"] = trial_df["Speaker"].astype(int)
trial_df["X"] = trial_df["IsAI"].astype(int)
trial_df["Y"] = trial_df["Model"].isin([2, 3]).astype(int)
trial_df["Correct"] = (trial_df["X"] == trial_df["Y"]).astype(int)
trial_df["Topic"] = ((trial_df["Artic"].astype(int) - 1) // 10 + 1).astype(int)

# Confidence to weight w in [0.5, 0.75, 1.0]
trial_df["w"] = (
    trial_df["Confidence"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
    .div(100.0)
)
trial_df["wacc_component"] = np.where(trial_df["Correct"] == 1, trial_df["w"], 1 - trial_df["w"])

# Familiarity rule
trial_df["Familiarity"] = np.where(
    ((trial_df["Institution"] == "Institution 2") & (trial_df["Speaker"] == 1))
    | ((trial_df["Institution"] == "Institution 1") & (trial_df["Speaker"].isin([3, 4]))),
    "Familiar",
    "Stranger",
)

# Speaker metadata
speaker_meta = {
    1: {"SpeakerName": "shi", "SpeakerGender": "Female", "SpeakerDevice": "Good"},
    2: {"SpeakerName": "yl", "SpeakerGender": "Male", "SpeakerDevice": "Poor"},
    3: {"SpeakerName": "zy", "SpeakerGender": "Male", "SpeakerDevice": "Good"},
    4: {"SpeakerName": "xsh", "SpeakerGender": "Female", "SpeakerDevice": "Poor"},
}
trial_df["SpeakerName"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerName"])
trial_df["SpeakerGender"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerGender"])
trial_df["SpeakerDevice"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerDevice"])

# Load participant metadata from questionnaire
q_path = DATA_DIR / "questionary.xlsx"
q_df = pd.read_excel(q_path)

participant_id_col = "4、您的受试者编号为？（*必答）"
participant_gender_col = "1、您的性别："
birth_year_col = "2、您的出生年份是？（*必答）"

q_df["ParticipantID"] = q_df[participant_id_col].astype(str).str.strip().str.upper()
q_df = q_df[q_df["ParticipantID"].str.match(r"^P\d+$", na=False)].copy()

# Participant gender mapping (fallback to raw code if unknown)
gender_map = {1: "Male", 2: "Female", "1": "Male", "2": "Female", "男": "Male", "女": "Female"}
q_df["ParticipantGender"] = q_df[participant_gender_col].map(gender_map).fillna(q_df[participant_gender_col].astype(str))

q_df["BirthYear"] = pd.to_numeric(q_df[birth_year_col], errors="coerce")
q_df["ParticipantAge"] = 2026 - q_df["BirthYear"]

# AI familiarity score = mean of questionnaire Q6-Q27 numeric responses
ai_cols = []
for col in q_df.columns:
    m = re.match(r"^(\d+)、", str(col))
    if m:
        q_idx = int(m.group(1))
        if 6 <= q_idx <= 21:
            ai_cols.append(col)

q_df[ai_cols] = q_df[ai_cols].apply(pd.to_numeric, errors="coerce")
q_df["AI_Familiarity"] = q_df[ai_cols].mean(axis=1)

# Auditory sensitivity score: mean of Q29-Q33
auditory_cols = []
for col in q_df.columns:
    m = re.match(r"^(\d+)、", str(col))
    if m:
        q_idx = int(m.group(1))
        if 29 <= q_idx <= 44:
            auditory_cols.append(col)

q_df[auditory_cols] = q_df[auditory_cols].apply(pd.to_numeric, errors="coerce")
q_df["Auditory_Sensitivity"] = q_df[auditory_cols].mean(axis=1)

participant_meta = q_df[[
    "ParticipantID", "ParticipantGender", "ParticipantAge", "AI_Familiarity", "Auditory_Sensitivity"
]].drop_duplicates("ParticipantID")
analysis_ext = trial_df.merge(participant_meta, on="ParticipantID", how="left")

print("Trial rows:", len(analysis_ext))
print("Participants in trials:", analysis_ext["ParticipantID"].nunique())
print("Participants matched in questionnaire:", analysis_ext["ParticipantGender"].notna().mean().round(3))
analysis_ext.head(3)

In [ ]:
def _safe_mean(masked_series: pd.Series) -> float:
    if len(masked_series) == 0:
        return np.nan
    return masked_series.mean()


def compute_metrics(df: pd.DataFrame) -> pd.Series:
    m1 = df[df["Model"] == 1]
    m2 = df[df["Model"] == 2]
    m3 = df[df["Model"] == 3]
    ai = df[df["Model"].isin([2, 3])]

    return pd.Series(
        {
            "n": len(df),
            "acc0": _safe_mean(df["Correct"]),
            "acc1": _safe_mean(m1["Correct"]),
            "acc2": _safe_mean(ai["Correct"]),
            "acc3": _safe_mean(m2["Correct"]),
            "acc4": _safe_mean(m3["Correct"]),
            "wacc0": _safe_mean(df["wacc_component"]),
        }
    )


def summarize_by(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = df.groupby(group_col, dropna=False).apply(compute_metrics).reset_index()
    return out.sort_values(group_col).reset_index(drop=True)


# 1) Institution
res_institution = summarize_by(analysis_ext, "Institution")[["Institution", "n", "acc0", "acc1", "acc3", "acc4", "wacc0"]]

# 2) Familiarity
res_familiarity = summarize_by(analysis_ext, "Familiarity")[["Familiarity", "n", "acc0", "acc1", "acc3", "acc4", "wacc0"]]

# 3) Speaker
res_speaker = summarize_by(analysis_ext, "Speaker")[["Speaker", "n", "acc0", "wacc0"]]

# 4) Speaker Gender
res_speaker_gender = summarize_by(analysis_ext, "SpeakerGender")[["SpeakerGender", "n", "acc0", "wacc0"]]

# 5) Participant Gender
res_participant_gender = summarize_by(analysis_ext.dropna(subset=["ParticipantGender"]), "ParticipantGender")[["ParticipantGender", "n", "acc0", "wacc0"]]

# 6) Participant Age (exact age)
age_df = analysis_ext.dropna(subset=["ParticipantAge"]).copy()
age_df["ParticipantAge"] = age_df["ParticipantAge"].astype(int)
res_participant_age = summarize_by(age_df, "ParticipantAge")[["ParticipantAge", "n", "acc0", "wacc0"]]

# 7) Topic (1-16)
res_topic = summarize_by(analysis_ext, "Topic")[["Topic", "n", "acc0", "wacc0"]]

# 8) AI Familiarity (binned)
ai_fam_df = analysis_ext.dropna(subset=["AI_Familiarity"]).copy()
ai_fam_df["AI_Familiarity_Bin"] = pd.cut(
    ai_fam_df["AI_Familiarity"],
    bins=[0, 2.5, 3.5, 4.5, 5.1],
    labels=["Low", "Mid", "High", "Very High"],
    include_lowest=True,
)
res_ai_fam = summarize_by(ai_fam_df, "AI_Familiarity_Bin")[["AI_Familiarity_Bin", "n", "acc0", "wacc0"]]

# 9) Auditory Sensitivity (binned)
aud_df = analysis_ext.dropna(subset=["Auditory_Sensitivity"]).copy()
aud_df["Auditory_Sensitivity_Bin"] = pd.cut(
    aud_df["Auditory_Sensitivity"],
    bins=[0, 2.5, 3.5, 4.5, 5.1],
    labels=["Low", "Mid", "High", "Very High"],
    include_lowest=True,
)
res_auditory = summarize_by(aud_df, "Auditory_Sensitivity_Bin")[["Auditory_Sensitivity_Bin", "n", "acc0", "wacc0"]]

# 10) Recording Equipment
res_equipment = summarize_by(analysis_ext, "SpeakerDevice")[["SpeakerDevice", "n", "acc0", "wacc0"]]

print("1) Institution")
display(res_institution)
print("2) Familiarity")
display(res_familiarity)
print("3) Speaker")
display(res_speaker)
print("4) Speaker Gender")
display(res_speaker_gender)
print("5) Participant Gender")
display(res_participant_gender)
print("6) Participant Age")
display(res_participant_age)
print("7) Topic")
display(res_topic)
print("8) AI Familiarity")
display(res_ai_fam)
print("9) Auditory Sensitivity")
display(res_auditory)
print("10) Recording Equipment")
display(res_equipment)

In [ ]:
# Age histogram
fig, ax = plt.subplots(figsize=(7, 4))
age_values = age_df["ParticipantAge"].dropna().astype(int)
bins = np.arange(age_values.min(), age_values.max() + 2) - 0.5
sns.histplot(age_values, bins=bins, ax=ax, color="#4C72B0")
ax.set_title("Participant Age Distribution")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.set_xticks(sorted(age_values.unique()))
plt.tight_layout()

In [ ]:
# Optional: collect all result tables in one dictionary
result_tables = {
    "institution": res_institution,
    "familiarity": res_familiarity,
    "speaker": res_speaker,
    "speaker_gender": res_speaker_gender,
    "participant_gender": res_participant_gender,
    "participant_age": res_participant_age,
    "topic": res_topic,
    "ai_familiarity": res_ai_fam,
    "auditory_sensitivity": res_auditory,
    "recording_equipment": res_equipment,
}

{k: v.shape for k, v in result_tables.items()}